What days to BTS, SFDPS and Eurocontrol have in common. So I can get the smallest set of raw ADS-B data possible. 

WHAT SFDPS days do I actually have? and what are the trends in them ?
Maybe I could download all the SFDPS data to my desktop and then I could delete it.  I need to calculate how much that would cost thought.... 

How to cheapely query what days of s3 data we have? 

Eurocontorl would be next easiset. 

# Eurocontrol sets

In [1]:
from pathlib import Path
import re

folder = Path("/Volumes/T2-SSD/planequery/data/raw/eurocontrol")

months = sorted(
    path.name
    for path in folder.iterdir()
    if path.is_dir() and re.fullmatch(r"\d{6}", path.name)
)

if months:
    print(f"Range: {months[0]} to {months[-1]}")
else:
    print("No YYYYMM folders found.")

Range: 201501 to 202509


In [ ]:
START_DATE = date(2024, 9, 1)
END_DATE = date(2025, 9, 1)  # Inclusive
# download adsblol and adsbx for this. algorithm segmentation

SFDPS...

In [ ]:
DATES = [date(2026,2,1), date(2026,3,1)] # SFDPS AND BTS can use this

In [ ]:
EUROCONTROL_TEST_DATES = [date(2025,4,1), date(2025,5,1), date(2025,6,1)]

In [4]:
from dataclasses import dataclass
from datetime import date
from typing import Any, Mapping

import polars as pl

from data_engineering.adsb.read_adsb import ADSBSource
from flights.flight_source import FlightSource
from flights.sources import FLIGHT_SOURCES_BY_NAME


@dataclass(frozen=True, slots=True)
class EvaluationStats:
    evaluation_date: date
    adsb_source: ADSBSource | None
    algorithm_source: FlightSource
    gold_source: FlightSource

    use_all_icaos: bool
    filter_rotorcraft: bool
    pia_or_american_ladd_only: bool

    true_positive: int
    false_positive: int
    false_negative: int

    takeoff_airport_match_count: int
    landing_airport_match_count: int
    airport_match_count: int

    @property
    def precision(self) -> float:
        denominator = self.true_positive + self.false_positive
        return self.true_positive / denominator if denominator else 0.0

    @property
    def recall(self) -> float:
        denominator = self.true_positive + self.false_negative
        return self.true_positive / denominator if denominator else 0.0

    @property
    def f1(self) -> float:
        denominator = self.precision + self.recall
        return (
            2 * self.precision * self.recall / denominator
            if denominator
            else 0.0
        )

    @property
    def takeoff_airport_match_pct(self) -> float:
        return self._airport_percentage(self.takeoff_airport_match_count)

    @property
    def landing_airport_match_pct(self) -> float:
        return self._airport_percentage(self.landing_airport_match_count)

    @property
    def airport_match_pct(self) -> float:
        return self._airport_percentage(self.airport_match_count)

    def _airport_percentage(self, match_count: int) -> float:
        return (
            match_count / self.true_positive
            if self.true_positive
            else 0.0
        )

    @classmethod
    def from_csv_row(
        cls,
        row: Mapping[str, Any],
    ) -> "EvaluationStats":
        if row["test_date"] == "average":
            raise ValueError(
                "Average rows do not contain actual integer counts"
            )

        return cls(
            evaluation_date=date.fromisoformat(row["test_date"]),
            adsb_source=parse_optional_adsb_source(row["adsb_src"]),
            algorithm_source=resolve_flight_source(row["test_src"]),
            gold_source=resolve_flight_source(row["gold_src"]),
            use_all_icaos=bool(row["use_all_icaos"]),
            filter_rotorcraft=bool(row["filter_rotorcraft"]),
            pia_or_american_ladd_only=bool(
                row["pia_or_american_ladd_only"]
            ),
            true_positive=parse_count(row["true_positive"]),
            false_positive=parse_count(row["false_positive"]),
            false_negative=parse_count(row["false_negative"]),
            takeoff_airport_match_count=parse_count(
                row["takeoff_airport_id_match_count"]
            ),
            landing_airport_match_count=parse_count(
                row["landing_airport_id_match_count"]
            ),
            airport_match_count=parse_count(
                row["airport_id_match_count"]
            ),
        )

def parse_count(value: int | float) -> int:
    """Convert CSV values such as 12457.0 into validated integer counts."""
    number = float(value)

    if not number.is_integer():
        raise ValueError(f"Expected an integer count, got {value!r}")

    return int(number)


def parse_optional_adsb_source(
    value: str | None,
) -> ADSBSource | None:
    if value is None or not value.strip():
        return None

    return ADSBSource(value)


def resolve_flight_source(name: str) -> FlightSource:
    try:
        return FLIGHT_SOURCES_BY_NAME[name]
    except KeyError:
        raise ValueError(f"Unknown flight source: {name!r}") from None

# Data Analysis

In [37]:
bts_csv_path = "/Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-03_20-56-20-039148_18884de_flights_evaluation.csv" 
sfpds_csv_path = "/Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-04_19-02-38-491861_18884de_flights_evaluation.csv"
eurocontrol_csv_path = "/Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-04_19-05-37-057977_bbf4675_flights_evaluation.csv"

In [38]:
import polars as pl
df = pl.read_csv(eurocontrol_csv_path)
stats = [
    EvaluationStats.from_csv_row(row)
    for row in df.filter(
        pl.col("test_date") != "average"
    ).to_dicts()
]

In [39]:
stats[0].evaluation_date

datetime.date(2025, 5, 1)

In [40]:
stats_by_day: dict[date, list] = dict()
for stat in stats:
    stats_by_day.setdefault(stat.evaluation_date, []).append(stat)

metrics_by_day = dict()

@dataclass
class DayMetrics:
    date: date
    adsblol_stats: EvaluationStats
    adsbx_stats: EvaluationStats

for date_key, stats in stats_by_day.items():
    dm = DayMetrics(
        date=date_key,
        adsblol_stats=next(
            stat for stat in stats if stat.adsb_source == ADSBSource.ADSBLOL
        ),
        adsbx_stats=next(
            stat for stat in stats if stat.adsb_source == ADSBSource.ADSBX
        ),
    )
    metrics_by_day[date_key] = dm

In [41]:
precision_diffs = []
recall_diffs = []
for dm in metrics_by_day.values():
    precision_diff = dm.adsblol_stats.precision - dm.adsbx_stats.precision
    recall_diff = dm.adsblol_stats.recall - dm.adsbx_stats.recall
    precision_diffs.append(precision_diff)
    recall_diffs.append(recall_diff)

print("Avg Precision Diff:", sum(precision_diffs) / len(precision_diffs))
print("Avg Recall Diff:", sum(recall_diffs) / len(recall_diffs))
    

Avg Precision Diff: -0.02570529158425444
Avg Recall Diff: -0.05773806783763963


In [42]:
# we could have class of stats

In [44]:
from flights.evaluation.cache import EvaluationCache

ec = EvaluationCache()
ec.list_entries()

test_date,adsb_src,test_src,gold_src,true_positive,false_positive,false_negative,precision,recall,f1,takeoff_airport_id_total_count,takeoff_airport_id_match_count,takeoff_airport_id_incorrect_match_count,takeoff_airport_id_match_pct,landing_airport_id_total_count,landing_airport_id_match_count,landing_airport_id_incorrect_match_count,landing_airport_id_match_pct,airport_id_total_count,airport_id_match_count,airport_id_incorrect_match_count,airport_id_match_pct,cache_version,test_source_version,test_source_config_fingerprint,gold_source_version,gold_source_config_fingerprint,adsb_parquet_version,airport_catalog_fingerprint,cache_path,cache_created_at
str,str,str,str,i64,i64,i64,f64,f64,f64,i64,i64,i64,f64,i64,i64,i64,f64,i64,i64,i64,f64,i32,i32,str,i32,str,i32,str,str,str
"""2024-06-01""","""adsblol""","""algorithm-nn""","""eurocontrol""",16140,6785,10353,0.704035,0.609218,0.653203,16140,15696,444,0.972491,16140,14899,1241,0.92311,16140,14494,1646,0.898017,3,4,"""61c4bb9d6b1c3085a14aa78cc24932…",4,"""1f8b04b10b9e61ee13da48857a4b5d…",6,"""0ce06fbe7cceb5e92e6474cb23f51b…","""/Volumes/T2-SSD/planequery/dat…","""2026-10-05T23:58:40.684193+00:…"
"""2026-03-01""","""adsblol""","""algorithm-nn""","""bts""",14762,3244,596,0.819838,0.961193,0.884906,14762,14741,21,0.998577,14762,14741,21,0.998577,14762,14720,42,0.997155,3,4,"""61c4bb9d6b1c3085a14aa78cc24932…",8,"""f4f583e303a9e55a2ff63f5083e214…",6,"""1f5eadc28dd11a340fcc45baf5b376…","""/Volumes/T2-SSD/planequery/dat…","""2026-10-05T23:34:57.865316+00:…"
"""2026-03-01""","""adsbx""","""algorithm-nn""","""bts""",15239,3393,146,0.817894,0.99051,0.895964,15239,15237,2,0.999869,15239,15236,3,0.999803,15239,15234,5,0.999672,3,4,"""61c4bb9d6b1c3085a14aa78cc24932…",8,"""f4f583e303a9e55a2ff63f5083e214…",2,"""1f5eadc28dd11a340fcc45baf5b376…","""/Volumes/T2-SSD/planequery/dat…","""2026-10-05T23:34:23.041054+00:…"


In [6]:
from flights.evaluation.evaluation import evaluate
from flights.sources import ALGORITHM_NN_SOURCE, BTS_SOURCE
from data_engineering.adsb.read_adsb import ADSBSource
from datetime import date

results = evaluate(test_source=ALGORITHM_NN_SOURCE, gold_source=BTS_SOURCE, test_dates=date(2026,3,1), adsb_source=ADSBSource.ADSBX)
results.stats

Using cached evaluation for 2026-03-01 algorithm-nn:bts (adsbx)
Stats for algorithm-nn:bts on 2026-03-01: {'true_positive': 15239, 'false_positive': 3393, 'false_negative': 146, 'takeoff_airport_id_total_count': 15239, 'takeoff_airport_id_match_count': 15237, 'takeoff_airport_id_incorrect_match_count': 2, 'landing_airport_id_total_count': 15239, 'landing_airport_id_match_count': 15236, 'landing_airport_id_incorrect_match_count': 3, 'airport_id_total_count': 15239, 'airport_id_match_count': 15234, 'airport_id_incorrect_match_count': 5, 'precision': 0.8178939458995277, 'recall': 0.990510237244069, 'f1': 0.8959637828144752, 'takeoff_airport_id_match_pct': 0.9998687577925061, 'landing_airport_id_match_pct': 0.9998031366887591, 'airport_id_match_pct': 0.9996718944812651}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-06_00-11-26-454912_36ca8be_flights_evaluation.csv


(EvaluationStats(test_date=datetime.date(2026, 3, 1), adsb_source=<ADSBSource.ADSBX: 'adsbx'>, test_source='algorithm-nn', gold_source='bts', true_positive=15239, false_positive=3393, false_negative=146, takeoff_airport_id_match_count=15237, landing_airport_id_match_count=15236, airport_id_match_count=15234),)